# 05 — Combined discovery-concordant results and figure

This notebook produces a labeled forest plot and summary tables for the combined VAE and univariate analyses.

The figure deliberately represents two estimands:

- **VAE and OLS targets:** the adjusted AoU disease main-effect coefficient from the common primary model.
- **Spline-discovery targets:** fixed age-specific adjusted case-control contrasts at ages 40, 55, and 70, but only when the prespecified **joint disease × spline(age)** test meets the plot threshold.

All in-range fixed-age contrasts are shown together. The notebook never selects an age because its individual contrast p-value is favorable. For spline rows, the plotted significance marker and row-selection p-value always come from the joint interaction test.

By default, rows are selected using discovery-concordant `p < 0.05`. Set `PLOT_P_THRESHOLD = 1.0` to show all successfully modeled targets, or set `REQUIRE_FDR10 = True` to require within-test-family, omic, and disease-domain FDR < 0.10.


In [ ]:
from pathlib import Path
import os
import sys

# Jupyter's file-browser directory and the kernel working directory can differ.
# Locate the project by its marker instead of trusting Path.cwd().
def find_project_root():
    marker_name = ".aou_combined_replication"
    project_name = "Combined_VAE_linear_AoU_replication"

    env_root = os.getenv("AOU_COMBINED_REPLICATION_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / marker_name).exists():
            return root
        raise FileNotFoundError(
            f"AOU_COMBINED_REPLICATION_ROOT={root} lacks {marker_name}"
        )

    here = Path.cwd().resolve()
    ancestors = [here, *here.parents]
    for candidate in ancestors:
        if (candidate / marker_name).exists():
            return candidate

    matches = []
    for parent in ancestors[:5]:
        patterns = [
            f"*/{marker_name}",
            f"{project_name}/{marker_name}",
            f"*/{project_name}/{marker_name}",
        ]
        for pattern in patterns:
            matches.extend(path.parent for path in parent.glob(pattern))

    matches = sorted(set(path.resolve() for path in matches))
    named = [path for path in matches if path.name == project_name]
    matches = named or matches
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(
        "Could not locate the combined replication project. Set "
        "AOU_COMBINED_REPLICATION_ROOT explicitly. Candidates found: "
        + str(matches)
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import project_config as cfg
from src import aou_replication_utils as aru

print("Kernel working directory:", Path.cwd().resolve())
print("Project root:", cfg.PROJECT_ROOT)
print("Workspace root:", cfg.WORKSPACE_ROOT)


In [ ]:
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

PRIMARY_FILE = cfg.RESULTS_DIR / "combined_target_replication_results_primary.tsv"
SPLINE_CONTRAST_FILE = (
    cfg.RESULTS_DIR / "combined_spline_age_specific_contrasts.tsv"
)
required = [PRIMARY_FILE, SPLINE_CONTRAST_FILE]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run notebook 04 first. Missing: " + "; ".join(str(path) for path in missing)
    )

primary = pd.read_csv(PRIMARY_FILE, sep="\t")
spline_contrasts = pd.read_csv(SPLINE_CONTRAST_FILE, sep="\t")

PLOT_P_THRESHOLD = 0.05
REQUIRE_FDR10 = False
MAX_TARGETS = None  # e.g. 80 caps targets before spline rows expand by age

plot_data = aru.build_discovery_concordant_forest_rows(
    primary=primary,
    spline_contrasts=spline_contrasts,
    p_threshold=PLOT_P_THRESHOLD,
    require_fdr10=REQUIRE_FDR10,
    max_targets=MAX_TARGETS,
)

modelled = primary["discovery_concordant_status"].eq("ok").sum()
selected_targets = plot_data["analysis_target_id"].nunique() if not plot_data.empty else 0
print(f"Discovery-concordant modelled targets: {modelled:,}")
print(f"Targets selected for plot:            {selected_targets:,}")
print(f"Forest-plot rows after age expansion:  {len(plot_data):,}")


In [ ]:
# Proper labels for discovery models and disease domains.
model_order = ["VAE_age", "VAE_resid", "OLS_age", "OLS_resid", "Spline"]
model_colors = {
    "VAE_age": "#1b9e77",
    "VAE_resid": "#66a61e",
    "OLS_age": "#7570b3",
    "OLS_resid": "#e7298a",
    "Spline": "#d95f02",
}
omic_markers = {
    "proteomics": "o",
    "chemistries": "s",
}

if not plot_data.empty:
    plot_data["model_order"] = pd.Categorical(
        plot_data["discovery_model"], categories=model_order, ordered=True
    )
    plot_data["omic_order"] = plot_data["omic_type"].map(
        {"proteomics": 0, "chemistries": 1}
    )
    plot_data["domain_order"] = plot_data["analysis_domain"].map(
        {
            "AD_dementia": 1,
            "Parkinsons": 2,
            "ALS_MND": 3,
            "CAD_coronary": 4,
            "CVD_composite": 5,
            "Heart_failure": 6,
            "T2D": 7,
        }
    ).fillna(99)


def relation_tag(row):
    if row["discovery_model"] == "Spline":
        return "joint age interaction"
    if row.get("plot_direction_relation") == "same_direction":
        return "direction matched"
    if row.get("plot_direction_relation") == "opposite_direction":
        return "opposite direction"
    return "direction not defined"


def build_row_label(row):
    age_suffix = ""
    if row["discovery_model"] == "Spline" and pd.notna(
        row.get("plot_reference_age_label")
    ):
        age_suffix = f" · {row['plot_reference_age_label']} contrast"
    return (
        f"{row['analysis_domain_label']} — {row['feature_display']} | "
        f"{row['discovery_model_label']} · {relation_tag(row)}{age_suffix}"
    )


if not plot_data.empty:
    plot_data["row_label"] = plot_data.apply(build_row_label, axis=1)
    plot_data = plot_data.sort_values(
        [
            "omic_order",
            "domain_order",
            "model_order",
            "plot_p",
            "feature_display",
            "plot_reference_age",
        ],
        ascending=[True, True, True, True, True, True],
        na_position="first",
    ).reset_index(drop=True)


In [ ]:
def draw_combined_forest(data):
    if data.empty:
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.text(
            0.5,
            0.5,
            "No discovery-concordant rows met the current plot threshold.",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.axis("off")
        return fig

    height = max(6, 0.42 * len(data) + 2.5)
    fig, ax = plt.subplots(figsize=(14, height))
    y = np.arange(len(data))[::-1]

    for i, row in data.iterrows():
        yi = y[i]
        color = model_colors.get(row["discovery_model"], "0.35")
        marker = omic_markers.get(row["omic_type"], "o")
        estimate = float(row["plot_estimate"])
        low = float(row["plot_ci_low"])
        high = float(row["plot_ci_high"])

        # Filled symbols indicate discovery/AoU directional agreement for VAE/OLS.
        # Spline discovery was an omnibus interaction test, so its fixed-age
        # contrasts remain open symbols rather than implying discovery direction.
        filled = (
            row["discovery_model"] != "Spline"
            and row.get("plot_direction_relation") == "same_direction"
        )
        face = color if filled else "white"
        ax.plot([low, high], [yi, yi], color=color, linewidth=1.5)
        ax.scatter(
            estimate,
            yi,
            marker=marker,
            s=52,
            facecolors=face,
            edgecolors=color,
            linewidths=1.4,
            zorder=3,
        )

        # For spline rows this q-value belongs to the joint interaction test,
        # not to the individual fixed-age contrast.
        if pd.notna(row["plot_q"]) and row["plot_q"] < 0.10:
            ax.text(high, yi, "  *", va="center", ha="left", color=color)

    ax.axvline(0, color="black", linewidth=0.9, linestyle="--")
    ax.set_yticks(y)
    ax.set_yticklabels(data["row_label"], fontsize=8.2)
    ax.set_xlabel("Adjusted standardized AoU case-control difference (95% CI)")
    ax.set_title(
        "AoU replication of VAE and univariate PRS–omic associations\n"
        "VAE/OLS: primary disease main effect; Spline: fixed-age contrasts "
        "selected by the joint disease × spline(age) test"
    )
    ax.grid(axis="x", alpha=0.2)

    # Add a separator between proteomics and chemistry sections.
    omic_change = data["omic_type"].ne(data["omic_type"].shift()).to_numpy()
    starts = np.where(omic_change)[0]
    for start in starts[1:]:
        separator_y = (y[start - 1] + y[start]) / 2
        ax.axhline(separator_y, color="0.75", linewidth=1)

    model_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=model_colors[model],
            markeredgecolor=model_colors[model],
            label=aru.DISCOVERY_MODEL_LABELS[model],
            markersize=7,
        )
        for model in model_order
        if model in set(data["discovery_model"])
    ]
    omic_handles = [
        Line2D(
            [0],
            [0],
            marker=marker,
            color="black",
            linestyle="none",
            markerfacecolor="white",
            label=aru.OMIC_LABELS[omic],
            markersize=7,
        )
        for omic, marker in omic_markers.items()
        if omic in set(data["omic_type"])
    ]
    symbol_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="0.35",
            markeredgecolor="0.35",
            label="Filled: VAE/OLS direction matched",
            markersize=7,
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="white",
            markeredgecolor="0.35",
            label="Open: opposite/undefined direction or spline contrast",
            markersize=7,
        ),
        Line2D(
            [0],
            [0],
            marker="$*$",
            color="0.35",
            linestyle="none",
            label="Within-family/domain FDR < 0.10",
            markersize=9,
        ),
    ]
    ax.legend(
        handles=model_handles + omic_handles + symbol_handles,
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        frameon=False,
    )
    fig.tight_layout()
    return fig


figure = draw_combined_forest(plot_data)
png = cfg.PLOTS_DIR / "combined_vae_univariate_discovery_concordant_forest.png"
pdf = cfg.PLOTS_DIR / "combined_vae_univariate_discovery_concordant_forest.pdf"
svg = cfg.PLOTS_DIR / "combined_vae_univariate_discovery_concordant_forest.svg"
figure.savefig(png, dpi=300, bbox_inches="tight")
figure.savefig(pdf, bbox_inches="tight")
figure.savefig(svg, bbox_inches="tight")
plt.show()

plot_data.to_csv(
    cfg.RESULTS_DIR / "combined_primary_forest_plot_rows.tsv",
    sep="\t",
    index=False,
)
print("Saved plot files:")
print(png)
print(pdf)
print(svg)


In [ ]:
# Denominator-aware comparison table. This is descriptive rather than an
# apples-to-apples estimate of method performance because feature availability,
# candidate generation, and the spline interaction test family differ.
method_summary_file = cfg.RESULTS_DIR / "primary_replication_summary_by_method.tsv"
main_effect_summary_file = (
    cfg.RESULTS_DIR / "primary_main_effect_replication_summary_by_method.tsv"
)
availability_file = cfg.RESULTS_DIR / "primary_target_availability_by_method.tsv"
method_summary = pd.read_csv(method_summary_file, sep="\t")
main_effect_summary = pd.read_csv(main_effect_summary_file, sep="\t")
availability = pd.read_csv(availability_file, sep="\t")

print("Discovery-concordant summary:")
display(method_summary)
print("Common main-effect summary:")
display(main_effect_summary)
print("Availability/status denominators:")
display(availability)

# Compact summary table retaining both the common main-effect estimate
# and the discovery-concordant evidence columns.
summary_cols = [
    "analysis_target_id",
    "discovery_framework",
    "discovery_model",
    "discovery_model_label",
    "omic_type",
    "omic_label",
    "analysis_domain",
    "analysis_domain_label",
    "feature_display",
    "uniprot",
    "gene_name",
    "prs_names",
    "discovery_direction",
    "replication_test_family",
    "beta_case",
    "se_case",
    "ci_low",
    "ci_high",
    "p_case",
    "q_within_analysis_omic_domain",
    "direction_relation",
    "replication_category",
    "interaction_statistic",
    "interaction_df",
    "p_interaction",
    "q_interaction_within_omic_domain",
    "observed_age_min",
    "observed_age_max",
    "discovery_concordant_p",
    "discovery_concordant_q",
    "discovery_concordant_q_scope",
    "discovery_concordant_replication_category",
    "discovery_concordant_n_cases",
    "discovery_concordant_n_controls",
    "discovery_concordant_status",
]
summary_table = primary[[c for c in summary_cols if c in primary.columns]].copy()
summary_table.to_csv(
    cfg.RESULTS_DIR / "combined_primary_results_summary.tsv",
    sep="\t",
    index=False,
)

# Spline contrasts are written separately as one row per fixed reference age.
# Their p-values are descriptive; the joint interaction p/q is the replication
# evidence used for target selection and figure significance markers.
spline_contrasts.to_csv(
    cfg.RESULTS_DIR / "combined_spline_age_specific_contrasts.tsv",
    sep="\t",
    index=False,
)
